# MicroPhaseLab Lesson 1：从 Polygon 到分割 Mask

本 Notebook 用合成数据解释第一版数据管线。先运行 `microphaselab demo`，再逐格运行。

## 学习目标

1. 区分 SEM 图像、polygon annotation 和 pixel mask。
2. 理解为什么 mask 只能包含 0/1。
3. 理解为何数据应按 sample 分组划分，而不是逐图随机划分。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

root = Path('../examples/demo')
manifest = pd.read_csv(root / 'processed/manifest.csv')
manifest

In [ ]:
row = manifest.iloc[0]
image = np.asarray(Image.open(row.image_path).convert('L'))
mask = np.asarray(Image.open(row.mask_path))
print('image shape:', image.shape)
print('mask values:', np.unique(mask))
print('MA area fraction:', mask.mean())

## 为什么不能只看 pixel accuracy？

若 MA 只占 5%，一个把所有像素都预测为背景的模型仍有 95% accuracy，却完全没有识别出 MA。后续版本会以 Dice、IoU、Precision 和 Recall 为主要指标。

## 阅读传统基线的结果

在运行 README 中的 `microphaselab baseline` 命令后，读取总体指标和逐图指标。合成演示中的亮区与标签被刻意设计得容易分割，所以高分只能说明流程连通，不能代表真实钢材数据的性能。

In [ ]:
import json

baseline_root = Path('../outputs/baseline/demo')
summary = json.loads((baseline_root / 'summary.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(baseline_root / 'metrics_per_image.csv')
summary_keys = [
    'mean_dice', 'mean_iou', 'mean_precision', 'mean_recall',
    'mean_area_fraction_absolute_error',
]
{key: summary[key] for key in summary_keys}

### 思考题

1. 哪张图的 Dice 最低？查看它的 prediction，判断误差主要来自假阳性还是假阴性。
2. 如果面积分数误差很小，但 IoU 很低，预测的边界可能发生了什么？
3. 为什么不能用这两张合成图的分数来选择真实数据的模型参数？